<a href="https://colab.research.google.com/github/ulinnuhagantengbgt-cmyk/Tugas-Data-Infrastruktur/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install schedule

In [2]:
import pandas as pd
import sqlite3
import json
import re
import schedule
import time
import logging
from datetime import datetime

In [4]:
from google.colab import files

uploaded = files.upload()

Saving siswa_sekolah_a.csv to siswa_sekolah_a.csv
Saving siswa_sekolah_b.sql to siswa_sekolah_b.sql
Saving siswa_sekolah_c.json to siswa_sekolah_c.json


In [5]:
CSV_FILE = "siswa_sekolah_a.csv"
SQL_FILE = "siswa_sekolah_b.sql"
JSON_FILE = "siswa_sekolah_c.json"

DB_FILE = "warehouse_siswa.db"

logging.basicConfig(
    filename="etl_log.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [6]:
def extract_csv():
    return pd.read_csv(CSV_FILE)


def extract_json():

    with open(JSON_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)

    df_json = pd.DataFrame(data)

    return df_json


def extract_sql():

    with open(SQL_FILE, "r", encoding="utf-8") as f:
        sql_text = f.read()

    pattern = r"\((.*?)\)"
    records = re.findall(pattern, sql_text)

    rows = []

    for record in records:
        rows.append(record.split(","))

    df_sql = pd.DataFrame(rows)

    return df_sql

In [7]:
def transform(df_csv, df_json, df_sql):

    # Sesuaikan nama kolom
    # Edit sesuai struktur dataset

    all_df = pd.concat(
        [df_csv, df_json, df_sql],
        ignore_index=True
    )

    all_df["source_file"] = "ETL"
    all_df["created_at"] = datetime.now()

    # hapus duplikat
    if "nisn" in all_df.columns:
        all_df = all_df.drop_duplicates(subset=["nisn"])

    # isi data kosong
    all_df = all_df.fillna("-")

    return all_df

In [13]:
df_csv = extract_csv()
df_json = extract_json()
df_sql = extract_sql()

final_df = transform(
    df_csv,
    df_json,
    df_sql
)

print(final_df.dtypes)
display(final_df.head())

nisn                      object
nama_siswa                object
jenis_kelamin             object
tanggal_lahir             object
sekolah                   object
alamat_sekolah            object
jenjang                   object
kota                      object
data_siswa                object
0                         object
1                         object
2                         object
3                         object
4                         object
5                         object
6                         object
7                         object
8                         object
9                         object
source_file               object
created_at        datetime64[us]
dtype: object


,nisn,nama_siswa,jenis_kelamin,tanggal_lahir,sekolah,alamat_sekolah,jenjang,kota,data_siswa,0,...,2,3,4,5,6,7,8,9,source_file,created_at
0,31234567.0,Andi Pratama,L,2015-03-10,SD Negeri 1 Semarang Barat,Jl. Mawar No.1,SD,Semarang,-,-,...,-,-,-,-,-,-,-,-,ETL,2026-06-05 00:44:40.830638
1,31234568.0,Sinta Dewi,P,2015-08-22,SD Negeri 1 Semarang Barat,Jl. Mawar No.1,SD,Semarang,-,-,...,-,-,-,-,-,-,-,-,ETL,2026-06-05 00:44:40.830638
2,31234569.0,Rizky Maulana,L,2010-11-05,SMP Negeri 2 Surakarta,Jl. Melati No.5,SMP,Surakarta,-,-,...,-,-,-,-,-,-,-,-,ETL,2026-06-05 00:44:40.830638
3,31234570.0,Dewi Lestari,P,2010-12-15,SMP Negeri 2 Surakarta,Jl. Melati No.5,SMP,Surakarta,-,-,...,-,-,-,-,-,-,-,-,ETL,2026-06-05 00:44:40.830638
4,-,-,-,-,-,-,-,-,"{'identitas': {'nisn': '0031234573', 'nama': '...",-,...,-,-,-,-,-,-,-,-,ETL,2026-06-05 00:44:40.830638


In [14]:
def extract_json():

    with open(JSON_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []

    for item in data["data_siswa"]:

        rows.append({
            "nisn": item["identitas"]["nisn"],
            "nama": item["identitas"]["nama"],
            "jenis_kelamin": item["identitas"]["jenis_kelamin"],
            "tanggal_lahir": item["identitas"]["tanggal_lahir"],
            "sekolah": item["sekolah"]["nama"],
            "alamat": item["sekolah"]["alamat"],
            "jenjang": item["sekolah"]["jenjang"],
            "kota": item["sekolah"]["kota"],
            "status": item["sekolah"]["status"]
        })

    return pd.DataFrame(rows)

In [15]:
df_sql = extract_sql()

print(df_sql.head())
print(df_sql.columns)

     0     1     2     3     4     5     6     7     8     9
0   11  None  None  None  None  None  None  None  None  None
1   15  None  None  None  None  None  None  None  None  None
2  100  None  None  None  None  None  None  None  None  None
3  'L'   'P'  None  None  None  None  None  None  None  None
4  100  None  None  None  None  None  None  None  None  None
RangeIndex(start=0, stop=10, step=1)


In [16]:
with open(SQL_FILE, "r", encoding="utf-8") as f:
    print(f.read()[:3000])

-- phpMyAdmin SQL Dump
-- version 5.2.1
-- https://www.phpmyadmin.net/
--
-- Host: 127.0.0.1
-- Generation Time: Oct 26, 2025 at 04:19 PM
-- Server version: 10.4.32-MariaDB
-- PHP Version: 8.0.30

SET SQL_MODE = "NO_AUTO_VALUE_ON_ZERO";
START TRANSACTION;
SET time_zone = "+00:00";


/*!40101 SET @OLD_CHARACTER_SET_CLIENT=@@CHARACTER_SET_CLIENT */;
/*!40101 SET @OLD_CHARACTER_SET_RESULTS=@@CHARACTER_SET_RESULTS */;
/*!40101 SET @OLD_COLLATION_CONNECTION=@@COLLATION_CONNECTION */;
/*!40101 SET NAMES utf8mb4 */;

--
-- Database: `sekolah_b`
--

-- --------------------------------------------------------

--
-- Table structure for table `siswa_sekolah_b`
--

CREATE TABLE `siswa_sekolah_b` (
  `id_mysql` int(11) NOT NULL,
  `nomor_induk` varchar(15) DEFAULT NULL,
  `nama_lengkap` varchar(100) DEFAULT NULL,
  `jk` enum('L','P') DEFAULT NULL,
  `tgl_lahir` date DEFAULT NULL,
  `nama_sekolah_asli` varchar(100) DEFAULT NULL,
  `alamat_sekolah_asli` text DEFAULT NULL,
  `tingkat` varchar(10) DEF

In [19]:
with open(SQL_FILE, "r", encoding="utf-8") as f:
    sql = f.read()

pos = sql.find("INSERT INTO")
print(sql[pos:pos+2500])

INSERT INTO `siswa_sekolah_b` (`id_mysql`, `nomor_induk`, `nama_lengkap`, `jk`, `tgl_lahir`, `nama_sekolah_asli`, `alamat_sekolah_asli`, `tingkat`, `kota_sekolah`, `tanggal_daftar`) VALUES
(1, 'SMP-001', 'Rina Wijaya', 'P', '2009-07-15', 'SMP PGRI 1 Semarang', 'Jl. Dahlia No.25', 'SMP', 'Semarang', '2025-10-26 13:28:14'),
(2, 'SMP-002', 'Fajar Nugroho', 'L', '2009-11-30', 'SMP PGRI 1 Semarang', 'Jl. Dahlia No.25', 'SMP', 'Semarang', '2025-10-26 13:28:14'),
(3, 'SMA-001', 'Lisa Permata', 'P', '2007-04-22', 'SMA Kristen 1 Surakarta', 'Jl. Sakura No.8', 'SMA', 'Surakarta', '2025-10-26 13:28:14'),
(4, 'SMA-002', 'Hendra Kurniawan', 'L', '2007-12-10', 'SMA Kristen 1 Surakarta', 'Jl. Sakura No.8', 'SMA', 'Surakarta', '2025-10-26 13:28:14');

--
-- Indexes for dumped tables
--

--
-- Indexes for table `siswa_sekolah_b`
--
ALTER TABLE `siswa_sekolah_b`
  ADD PRIMARY KEY (`id_mysql`);

--
-- AUTO_INCREMENT for dumped tables
--

--
-- AUTO_INCREMENT for table `siswa_sekolah_b`
--
ALTER TABLE `si

In [20]:
def extract_sql():

    with open(SQL_FILE, "r", encoding="utf-8") as f:
        sql = f.read()

    pattern = r"\((\d+),\s*'([^']*)',\s*'([^']*)',\s*'([^']*)',\s*'([^']*)',\s*'([^']*)',\s*'([^']*)',\s*'([^']*)',\s*'([^']*)',\s*'([^']*)'\)"

    matches = re.findall(pattern, sql)

    rows = []

    for m in matches:
        rows.append({
            "nisn": m[1],                 # nomor_induk
            "nama": m[2],                 # nama_lengkap
            "jenis_kelamin": m[3],        # jk
            "tanggal_lahir": m[4],        # tgl_lahir
            "sekolah": m[5],              # nama_sekolah_asli
            "alamat": m[6],               # alamat_sekolah_asli
            "jenjang": m[7],              # tingkat
            "kota": m[8],                 # kota_sekolah
            "status": "Aktif"
        })

    return pd.DataFrame(rows)

In [21]:
def transform(df_csv, df_json, df_sql):

    df_csv = df_csv.rename(columns={
        "nama_siswa": "nama",
        "alamat_sekolah": "alamat"
    })

    df_csv["status"] = "Aktif"

    df_csv["source_file"] = "CSV"
    df_json["source_file"] = "JSON"
    df_sql["source_file"] = "SQL"

    final_df = pd.concat(
        [df_csv, df_json, df_sql],
        ignore_index=True
    )

    final_df["created_at"] = datetime.now()

    final_df = final_df.drop_duplicates(subset=["nisn"])

    final_df = final_df.fillna("-")

    return final_df

In [22]:
df_csv = extract_csv()
df_json = extract_json()
df_sql = extract_sql()

final_df = transform(df_csv, df_json, df_sql)

print(final_df.columns)
display(final_df.head())

Index(['nisn', 'nama', 'jenis_kelamin', 'tanggal_lahir', 'sekolah', 'alamat',
       'jenjang', 'kota', 'status', 'source_file', 'created_at'],
      dtype='object')


,nisn,nama,jenis_kelamin,tanggal_lahir,sekolah,alamat,jenjang,kota,status,source_file,created_at
0,31234567,Andi Pratama,L,2015-03-10,SD Negeri 1 Semarang Barat,Jl. Mawar No.1,SD,Semarang,Aktif,CSV,2026-06-05 00:48:47.668635
1,31234568,Sinta Dewi,P,2015-08-22,SD Negeri 1 Semarang Barat,Jl. Mawar No.1,SD,Semarang,Aktif,CSV,2026-06-05 00:48:47.668635
2,31234569,Rizky Maulana,L,2010-11-05,SMP Negeri 2 Surakarta,Jl. Melati No.5,SMP,Surakarta,Aktif,CSV,2026-06-05 00:48:47.668635
3,31234570,Dewi Lestari,P,2010-12-15,SMP Negeri 2 Surakarta,Jl. Melati No.5,SMP,Surakarta,Aktif,CSV,2026-06-05 00:48:47.668635
4,0031234573,Ahmad Fauzi,L,2014-02-18,SD Islam Terpadu Nurul Fikri,"Jl. Cempaka No.30, Semarang",SD,Semarang,Swasta,JSON,2026-06-05 00:48:47.668635


In [23]:
def load_sqlite(df):

    conn = sqlite3.connect("warehouse_siswa.db")

    df.to_sql(
        "master_siswa",
        conn,
        if_exists="replace",
        index=False
    )

    conn.close()

    print("Data berhasil masuk ke SQLite")

In [24]:
def etl_process():

    try:

        df_csv = extract_csv()
        df_json = extract_json()
        df_sql = extract_sql()

        final_df = transform(
            df_csv,
            df_json,
            df_sql
        )

        load_sqlite(final_df)

        logging.info(
            f"SUCCESS | Total Data : {len(final_df)}"
        )

        print("ETL SUCCESS")

        return final_df

    except Exception as e:

        logging.error(str(e))

        print("ETL FAILED :", e)

        return None

In [25]:
hasil = etl_process()

if hasil is not None:
    display(hasil.head())

Data berhasil masuk ke SQLite
ETL SUCCESS


,nisn,nama,jenis_kelamin,tanggal_lahir,sekolah,alamat,jenjang,kota,status,source_file,created_at
0,31234567,Andi Pratama,L,2015-03-10,SD Negeri 1 Semarang Barat,Jl. Mawar No.1,SD,Semarang,Aktif,CSV,2026-06-05 00:50:33.695170
1,31234568,Sinta Dewi,P,2015-08-22,SD Negeri 1 Semarang Barat,Jl. Mawar No.1,SD,Semarang,Aktif,CSV,2026-06-05 00:50:33.695170
2,31234569,Rizky Maulana,L,2010-11-05,SMP Negeri 2 Surakarta,Jl. Melati No.5,SMP,Surakarta,Aktif,CSV,2026-06-05 00:50:33.695170
3,31234570,Dewi Lestari,P,2010-12-15,SMP Negeri 2 Surakarta,Jl. Melati No.5,SMP,Surakarta,Aktif,CSV,2026-06-05 00:50:33.695170
4,0031234573,Ahmad Fauzi,L,2014-02-18,SD Islam Terpadu Nurul Fikri,"Jl. Cempaka No.30, Semarang",SD,Semarang,Swasta,JSON,2026-06-05 00:50:33.695170


In [26]:
conn = sqlite3.connect("warehouse_siswa.db")

cek = pd.read_sql(
    "SELECT * FROM master_siswa",
    conn
)

display(cek)

conn.close()

,nisn,nama,jenis_kelamin,tanggal_lahir,sekolah,alamat,jenjang,kota,status,source_file,created_at
0,31234567,Andi Pratama,L,2015-03-10,SD Negeri 1 Semarang Barat,Jl. Mawar No.1,SD,Semarang,Aktif,CSV,2026-06-05 00:50:33.695170
1,31234568,Sinta Dewi,P,2015-08-22,SD Negeri 1 Semarang Barat,Jl. Mawar No.1,SD,Semarang,Aktif,CSV,2026-06-05 00:50:33.695170
2,31234569,Rizky Maulana,L,2010-11-05,SMP Negeri 2 Surakarta,Jl. Melati No.5,SMP,Surakarta,Aktif,CSV,2026-06-05 00:50:33.695170
3,31234570,Dewi Lestari,P,2010-12-15,SMP Negeri 2 Surakarta,Jl. Melati No.5,SMP,Surakarta,Aktif,CSV,2026-06-05 00:50:33.695170
4,0031234573,Ahmad Fauzi,L,2014-02-18,SD Islam Terpadu Nurul Fikri,"Jl. Cempaka No.30, Semarang",SD,Semarang,Swasta,JSON,2026-06-05 00:50:33.695170
5,0031234574,Sari Indah Pertiwi,P,2014-09-05,SD Islam Terpadu Nurul Fikri,"Jl. Cempaka No.30, Semarang",SD,Semarang,Swasta,JSON,2026-06-05 00:50:33.695170
6,0031234575,Dimas Prayogo,L,2008-08-12,SMP Santa Maria Semarang,"Jl. Flamboyan No.12, Semarang",SMP,Semarang,Swasta,JSON,2026-06-05 00:50:33.695170
7,SMP-001,Rina Wijaya,P,2009-07-15,SMP PGRI 1 Semarang,Jl. Dahlia No.25,SMP,Semarang,Aktif,SQL,2026-06-05 00:50:33.695170
8,SMP-002,Fajar Nugroho,L,2009-11-30,SMP PGRI 1 Semarang,Jl. Dahlia No.25,SMP,Semarang,Aktif,SQL,2026-06-05 00:50:33.695170
9,SMA-001,Lisa Permata,P,2007-04-22,SMA Kristen 1 Surakarta,Jl. Sakura No.8,SMA,Surakarta,Aktif,SQL,2026-06-05 00:50:33.695170


In [27]:
logging.basicConfig(
    filename="etl_log.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [29]:
import logging

logging.basicConfig(
    filename="etl_log.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [30]:
logging.info("Test Log")
print("Log berhasil ditulis")

Log berhasil ditulis


In [31]:
import os

print(os.listdir())

['.config', 'siswa_sekolah_c.json', 'warehouse_siswa.db', 'source_uas (1).zip', 'siswa_sekolah_b.sql', 'siswa_sekolah_a.csv', 'sample_data']


In [33]:
import logging

logging.basicConfig(
    filename="etl_log.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)

logging.info("ETL pertama berhasil")

print("Log dibuat")

Log dibuat


In [34]:
import os
print(os.listdir())

['.config', 'siswa_sekolah_c.json', 'warehouse_siswa.db', 'etl_log.log', 'source_uas (1).zip', 'siswa_sekolah_b.sql', 'siswa_sekolah_a.csv', 'sample_data']


In [35]:
with open("etl_log.log", "r") as f:
    print(f.read())

2026-06-05 00:52:46,705 - INFO - ETL pertama berhasil



In [36]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("warehouse_siswa.db")

df = pd.read_sql(
    "SELECT * FROM master_siswa",
    conn
)

display(df)

conn.close()

,nisn,nama,jenis_kelamin,tanggal_lahir,sekolah,alamat,jenjang,kota,status,source_file,created_at
0,31234567,Andi Pratama,L,2015-03-10,SD Negeri 1 Semarang Barat,Jl. Mawar No.1,SD,Semarang,Aktif,CSV,2026-06-05 00:50:33.695170
1,31234568,Sinta Dewi,P,2015-08-22,SD Negeri 1 Semarang Barat,Jl. Mawar No.1,SD,Semarang,Aktif,CSV,2026-06-05 00:50:33.695170
2,31234569,Rizky Maulana,L,2010-11-05,SMP Negeri 2 Surakarta,Jl. Melati No.5,SMP,Surakarta,Aktif,CSV,2026-06-05 00:50:33.695170
3,31234570,Dewi Lestari,P,2010-12-15,SMP Negeri 2 Surakarta,Jl. Melati No.5,SMP,Surakarta,Aktif,CSV,2026-06-05 00:50:33.695170
4,0031234573,Ahmad Fauzi,L,2014-02-18,SD Islam Terpadu Nurul Fikri,"Jl. Cempaka No.30, Semarang",SD,Semarang,Swasta,JSON,2026-06-05 00:50:33.695170
5,0031234574,Sari Indah Pertiwi,P,2014-09-05,SD Islam Terpadu Nurul Fikri,"Jl. Cempaka No.30, Semarang",SD,Semarang,Swasta,JSON,2026-06-05 00:50:33.695170
6,0031234575,Dimas Prayogo,L,2008-08-12,SMP Santa Maria Semarang,"Jl. Flamboyan No.12, Semarang",SMP,Semarang,Swasta,JSON,2026-06-05 00:50:33.695170
7,SMP-001,Rina Wijaya,P,2009-07-15,SMP PGRI 1 Semarang,Jl. Dahlia No.25,SMP,Semarang,Aktif,SQL,2026-06-05 00:50:33.695170
8,SMP-002,Fajar Nugroho,L,2009-11-30,SMP PGRI 1 Semarang,Jl. Dahlia No.25,SMP,Semarang,Aktif,SQL,2026-06-05 00:50:33.695170
9,SMA-001,Lisa Permata,P,2007-04-22,SMA Kristen 1 Surakarta,Jl. Sakura No.8,SMA,Surakarta,Aktif,SQL,2026-06-05 00:50:33.695170


In [37]:
import pandas as pd

df = pd.read_csv("siswa_sekolah_a.csv")

data_baru = {
    "nisn": "0031234999",
    "nama_siswa": "Budi Santoso",
    "jenis_kelamin": "L",
    "tanggal_lahir": "2015-01-01",
    "sekolah": "SD Negeri 1 Semarang Barat",
    "alamat_sekolah": "Jl. Mawar No.1",
    "jenjang": "SD",
    "kota": "Semarang"
}

df.loc[len(df)] = data_baru

df.to_csv("siswa_sekolah_a.csv", index=False)

print("Data baru berhasil ditambahkan")

Data baru berhasil ditambahkan


In [38]:
hasil = etl_process()

Data berhasil masuk ke SQLite
ETL SUCCESS


In [39]:
conn = sqlite3.connect("warehouse_siswa.db")

cek = pd.read_sql(
    "SELECT * FROM master_siswa WHERE nisn='0031234999'",
    conn
)

display(cek)

conn.close()

,nisn,nama,jenis_kelamin,tanggal_lahir,sekolah,alamat,jenjang,kota,status,source_file,created_at


In [40]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("warehouse_siswa.db")

cek = pd.read_sql(
    "SELECT * FROM master_siswa",
    conn
)

print("Jumlah data:", len(cek))
display(cek)

conn.close()

Jumlah data: 12


,nisn,nama,jenis_kelamin,tanggal_lahir,sekolah,alamat,jenjang,kota,status,source_file,created_at
0,31234567,Andi Pratama,L,2015-03-10,SD Negeri 1 Semarang Barat,Jl. Mawar No.1,SD,Semarang,Aktif,CSV,2026-06-05 00:54:22.020765
1,31234568,Sinta Dewi,P,2015-08-22,SD Negeri 1 Semarang Barat,Jl. Mawar No.1,SD,Semarang,Aktif,CSV,2026-06-05 00:54:22.020765
2,31234569,Rizky Maulana,L,2010-11-05,SMP Negeri 2 Surakarta,Jl. Melati No.5,SMP,Surakarta,Aktif,CSV,2026-06-05 00:54:22.020765
3,31234570,Dewi Lestari,P,2010-12-15,SMP Negeri 2 Surakarta,Jl. Melati No.5,SMP,Surakarta,Aktif,CSV,2026-06-05 00:54:22.020765
4,31234999,Budi Santoso,L,2015-01-01,SD Negeri 1 Semarang Barat,Jl. Mawar No.1,SD,Semarang,Aktif,CSV,2026-06-05 00:54:22.020765
5,0031234573,Ahmad Fauzi,L,2014-02-18,SD Islam Terpadu Nurul Fikri,"Jl. Cempaka No.30, Semarang",SD,Semarang,Swasta,JSON,2026-06-05 00:54:22.020765
6,0031234574,Sari Indah Pertiwi,P,2014-09-05,SD Islam Terpadu Nurul Fikri,"Jl. Cempaka No.30, Semarang",SD,Semarang,Swasta,JSON,2026-06-05 00:54:22.020765
7,0031234575,Dimas Prayogo,L,2008-08-12,SMP Santa Maria Semarang,"Jl. Flamboyan No.12, Semarang",SMP,Semarang,Swasta,JSON,2026-06-05 00:54:22.020765
8,SMP-001,Rina Wijaya,P,2009-07-15,SMP PGRI 1 Semarang,Jl. Dahlia No.25,SMP,Semarang,Aktif,SQL,2026-06-05 00:54:22.020765
9,SMP-002,Fajar Nugroho,L,2009-11-30,SMP PGRI 1 Semarang,Jl. Dahlia No.25,SMP,Semarang,Aktif,SQL,2026-06-05 00:54:22.020765


In [42]:
import schedule
import time

schedule.every(1).minutes.do(etl_process)

print("Scheduler aktif...")

Scheduler aktif...


In [44]:
while True:
    schedule.run_pending()
    time.sleep(1)

Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL SUCCESS
Data berhasil masuk ke SQLite
ETL 

KeyboardInterrupt: 

In [45]:
import pandas as pd

df = pd.read_csv("siswa_sekolah_a.csv")

data_baru = {
    "nisn": "0031234999",
    "nama_siswa": "Budi Santoso",
    "jenis_kelamin": "L",
    "tanggal_lahir": "2015-01-01",
    "sekolah": "SD Negeri 1 Semarang Barat",
    "alamat_sekolah": "Jl. Mawar No.1",
    "jenjang": "SD",
    "kota": "Semarang"
}

df.loc[len(df)] = data_baru

df.to_csv("siswa_sekolah_a.csv", index=False)

print("Data baru berhasil ditambahkan")

Data baru berhasil ditambahkan


In [46]:
hasil = etl_process()

Data berhasil masuk ke SQLite
ETL SUCCESS


In [47]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("warehouse_siswa.db")

cek = pd.read_sql(
    "SELECT * FROM master_siswa WHERE nisn='0031234999'",
    conn
)

print(cek)

conn.close()

Empty DataFrame
Columns: [nisn, nama, jenis_kelamin, tanggal_lahir, sekolah, alamat, jenjang, kota, status, source_file, created_at]
Index: []


In [48]:
with open("etl_log.log", "r") as f:
    print(f.read())

2026-06-05 00:52:46,705 - INFO - ETL pertama berhasil
2026-06-05 00:54:22,042 - INFO - SUCCESS | Total Data : 12
2026-06-05 00:57:45,881 - INFO - SUCCESS | Total Data : 12
2026-06-05 00:58:38,554 - INFO - SUCCESS | Total Data : 12
2026-06-05 00:58:46,580 - INFO - SUCCESS | Total Data : 12
2026-06-05 00:59:38,613 - INFO - SUCCESS | Total Data : 12
2026-06-05 00:59:46,640 - INFO - SUCCESS | Total Data : 12
2026-06-05 01:00:38,674 - INFO - SUCCESS | Total Data : 12
2026-06-05 01:00:46,701 - INFO - SUCCESS | Total Data : 12
2026-06-05 01:01:38,732 - INFO - SUCCESS | Total Data : 12
2026-06-05 01:01:46,755 - INFO - SUCCESS | Total Data : 12
2026-06-05 01:02:38,786 - INFO - SUCCESS | Total Data : 12
2026-06-05 01:02:46,812 - INFO - SUCCESS | Total Data : 12
2026-06-05 01:03:38,848 - INFO - SUCCESS | Total Data : 12
2026-06-05 01:03:46,875 - INFO - SUCCESS | Total Data : 12
2026-06-05 01:04:38,907 - INFO - SUCCESS | Total Data : 12
2026-06-05 01:04:46,930 - INFO - SUCCESS | Total Data : 12
20

# Implementasi ETL Multi Format Menggunakan Python dan SQLite pada Data Siswa

## Pendahuluan

Dalam era digital saat ini, data menjadi aset penting bagi organisasi maupun instansi pendidikan. Data sering kali berasal dari berbagai sumber dengan format yang berbeda-beda sehingga diperlukan proses integrasi agar dapat digunakan secara efektif. Salah satu metode yang digunakan untuk mengintegrasikan data adalah proses ETL (Extract, Transform, Load).

Pada tugas Ujian Akhir Semester (UAS) mata kuliah Data Infrastructure and Analytics, dilakukan pembangunan sistem ETL sederhana menggunakan Python untuk mengolah data siswa yang berasal dari tiga sumber berbeda, yaitu file CSV, SQL Dump, dan JSON. Hasil integrasi data kemudian disimpan ke dalam database SQLite serta dilengkapi dengan scheduler dan monitoring log.

## Tujuan Praktik

Adapun tujuan dari praktik ini adalah:

1. Membaca data dari berbagai format file.
2. Melakukan proses transformasi dan integrasi data.
3. Menyimpan data hasil integrasi ke database SQLite.
4. Mengotomatisasi proses ETL menggunakan scheduler.
5. Melakukan monitoring proses ETL menggunakan logging.

## Dataset yang Digunakan

Dataset yang digunakan terdiri dari tiga sumber data, yaitu:

1. siswa_sekolah_a.csv
2. siswa_sekolah_b.sql
3. siswa_sekolah_c.json

Masing-masing file memiliki struktur data yang berbeda sehingga perlu dilakukan proses standarisasi sebelum digabungkan.

## Tahap Extract

Tahap Extract merupakan proses pengambilan data dari berbagai sumber.

### Extract Data CSV

Data dari file CSV dibaca menggunakan library pandas sehingga menghasilkan dataframe yang berisi informasi siswa seperti NISN, nama siswa, jenis kelamin, tanggal lahir, sekolah, alamat sekolah, jenjang, dan kota.

### Extract Data JSON

Data JSON memiliki struktur nested yang terdiri dari bagian identitas dan sekolah. Oleh karena itu dilakukan parsing data agar setiap atribut dapat diubah menjadi format tabular yang mudah diproses.

### Extract Data SQL Dump

Data SQL Dump diekstrak menggunakan teknik regular expression (regex) untuk mengambil data yang terdapat pada perintah INSERT INTO. Data yang berhasil diekstrak kemudian dikonversi menjadi dataframe.

## Tahap Transform

Setelah seluruh data berhasil diekstrak, dilakukan proses transformasi agar seluruh sumber memiliki struktur yang sama.

Schema target yang digunakan adalah:

* nisn
* nama
* jenis_kelamin
* tanggal_lahir
* sekolah
* alamat
* jenjang
* kota
* status

Selain itu dilakukan beberapa proses tambahan, yaitu:

1. Menyamakan nama kolom dari setiap sumber data.
2. Menambahkan kolom source_file untuk mengetahui asal data.
3. Menambahkan kolom created_at sebagai waktu proses ETL.
4. Menghapus data duplikat berdasarkan NISN.
5. Mengisi data kosong menggunakan nilai default apabila diperlukan.

Setelah proses transformasi selesai, seluruh data digabungkan menjadi satu dataframe utama.

## Tahap Load

Tahap Load dilakukan dengan menyimpan data hasil transformasi ke dalam database SQLite.

Database yang digunakan bernama:

warehouse_siswa.db

Sedangkan tabel utama yang dibuat adalah:

master_siswa

Penyimpanan data dilakukan secara otomatis menggunakan Python sehingga data hasil ETL dapat langsung digunakan untuk proses analisis selanjutnya.

## Implementasi Scheduler

Agar proses ETL dapat berjalan secara otomatis, digunakan library schedule.

Scheduler dikonfigurasi untuk menjalankan proses ETL setiap satu menit sebagai simulasi otomatisasi proses data engineering.

Dengan adanya scheduler, sistem dapat membaca perubahan data tanpa perlu menjalankan proses ETL secara manual.

## Logging dan Monitoring

Monitoring dilakukan menggunakan library logging.

Setiap proses ETL yang berhasil maupun gagal akan dicatat ke dalam file:

etl_log.log

Informasi yang dicatat meliputi:

* Waktu proses ETL
* Status proses
* Jumlah data yang diproses
* Informasi error apabila terjadi kegagalan

Dengan adanya logging, proses monitoring dan debugging menjadi lebih mudah dilakukan.

## Simulasi Penambahan Data Baru

Sebagai pengujian akhir, dilakukan penambahan data baru ke dalam file CSV.

Setelah data baru ditambahkan, scheduler menjalankan kembali proses ETL dan data baru berhasil masuk ke dalam tabel master_siswa pada database SQLite tanpa perlu melakukan input manual ke database.

Hasil ini menunjukkan bahwa pipeline ETL yang dibangun telah bekerja dengan baik dan mampu menangani perubahan data secara otomatis.

## Hasil Pengujian

Berdasarkan hasil pengujian yang dilakukan, diperoleh hasil sebagai berikut:

1. Data dari tiga sumber berbeda berhasil dibaca dengan baik.
2. Proses transformasi berhasil menyamakan struktur data.
3. Data berhasil disimpan ke dalam database SQLite.
4. Scheduler berhasil menjalankan ETL secara otomatis.
5. Logging berhasil mencatat aktivitas ETL.
6. Data baru dapat terintegrasi secara otomatis ke dalam database.

## Kesimpulan

Berdasarkan praktik yang telah dilakukan, dapat disimpulkan bahwa proses ETL menggunakan Python berhasil diimplementasikan untuk mengintegrasikan data siswa dari berbagai format file. Data yang berasal dari CSV, SQL Dump, dan JSON berhasil diekstrak, ditransformasi, dan dimuat ke dalam database SQLite.

Selain itu, penggunaan scheduler dan logging mampu meningkatkan otomatisasi serta kemudahan monitoring proses ETL. Sistem yang dibangun telah memenuhi kebutuhan integrasi data sederhana dan dapat menjadi dasar dalam pengembangan sistem data engineering yang lebih kompleks di masa mendatang.
